# Carnet 3 : L'Automatisation du Tuning avec MLflow

Dans le carnet précédent, nous avons vu que tester des hyperparamètres à la main ou avec une boucle aléatoire naïve était fastidieux, illisible dans MLflow et surtout sous-optimal.

L'objectif de ce carnet est de laisser l'ordinateur trouver la configuration parfaite lui-même. Nous allons comparer deux bibliothèques très connues pour optimiser notre modèle XGBoost :
- **GridSearchCV** : L'approche classique (exhaustive mais lente).
- **Optuna** : Un framework moderne, intelligent (optimisation bayésienne), extrêmement rapide et qui s'intègre parfaitement avec MLflow.

Pour rappel, notre critère métier principal est le **Recall** (pour rater le moins d'accidents graves possible).

## 1. Imports et Chargement des Données

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, log_loss, confusion_matrix
import xgboost as xgb
from xgboost import XGBClassifier

import mlflow
import mlflow.sklearn
import optuna
from optuna.integration.mlflow import MLflowCallback
from optuna.visualization import plot_optimization_history, plot_param_importances

# Configuration de base MLflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Chargement et préparation habituelle
df_accident = pd.read_csv('data/dataset_accident.csv', sep=';' )
y = df_accident["grav_binary"]
X = df_accident.drop(columns=["grav_ordered", "grav_binary"])

# Nouveauté : On ajoute 'stratify=y' pour garantir la même proportion d'accidents graves 
# dans le train et le test. Très utile pour l'optimisation avancée !
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

## 2. Approche 1 : GridSearchCV

C'est la méthode "force brute". On lui donne une grille de valeurs, et elle va tester absolument toutes les combinaisons possibles. 

L'avantage de `mlflow.sklearn.autolog()` est qu'il tracke tout automatiquement pendant la recherche de grille !

In [ ]:
mlflow.set_experiment("Optimisation_XGBoost_Recall_GridSearch")

# Activation de l'autologging pour GridSearchCV
mlflow.sklearn.autolog()

param_grid = {
    'n_estimators': [100, 200, 500],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 9],
    'subsample': [0.7, 1.0]
}

print("Grille définie : 3 x 3 x 3 x 2 = 54 combinaisons à tester.")
print("Avec une validation croisée de 3 (cv=3), cela fait 162 entraînements de modèles. Soyez patient...")

with mlflow.start_run(run_name="GridSearch_Execution"):
    xgb_model = XGBClassifier(eval_metric="logloss")
    
    grid_search = GridSearchCV(
        estimator=xgb_model,
        param_grid=param_grid,
        cv=3,
        scoring='recall', # Notre métrique star !
        verbose=1,
        n_jobs=-1 # Utilise tous les coeurs du processeur
    )

    grid_search.fit(X_train, y_train)
    
    # Log manuel (bien que l'autologging fasse déjà le gros du travail)
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metric("best_recall", grid_search.best_score_)
    
    # Sauvegarde du modèle sous un nom distinctif
    mlflow.xgboost.log_model(grid_search.best_estimator_, "XGBoost_GridSearch_Best")
 
    print("\n--- Résultats GridSearchCV ---")
    print(f"Meilleur recall en CV : {grid_search.best_score_:.4f}")
    print(f"Meilleurs paramètres  : {grid_search.best_params_}")

## 3. Approche 2 : Optuna

GridSearch est long... Optuna est intelligent ! Au lieu de quadriller, Optuna va essayer une configuration, voir le résultat, et ajuster mathématiquement sa prochaine supposition (Optimisation Bayésienne).

De plus, nous pouvons utiliser `MLflowCallback` très simplement pour que chaque itération (ou *trial*) d'Optuna atterrisse joliment dans notre dashboard MLflow en tant qu'enfant (nested run).

In [ ]:
# Désactivation de l'autolog pour garder un MLflow propre avec Optuna
mlflow.sklearn.autolog(disable=True) 

experiment_name = "Optimisation_XGBoost_Optuna"
mlflow.set_experiment(experiment_name)
exp = mlflow.get_experiment_by_name(experiment_name)

def objective(trial):
    # Optuna 'suggest' les paramètres de façon large et continue, sans grille fixe
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'eval_metric': 'logloss'
    }
    
    model = xgb.XGBClassifier(**params, random_state=42)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    return recall_score(y_test, preds)


with mlflow.start_run(run_name="Optuna_Master_Run"):
    
    # Un callback surpuissant qui trace l'historique d'Optuna directement dans MLflow
    mlflc = MLflowCallback(
        tracking_uri=mlflow.get_tracking_uri(),
        metric_name="recall",
        mlflow_kwargs={"experiment_id": exp.experiment_id, 'nested': True}
    )

    # direction="maximize" car on veut augmenter le Recall (direction="minimize" pour Log Loss par exemple)
    study = optuna.create_study(direction="maximize")
    print("Lancement d'Optuna. Il va faire 20 essais avec une intelligence bayésienne.")
    study.optimize(objective, n_trials=20, callbacks=[mlflc])

    # --- Fin de l'étude, on logge le champion ! ---
    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_recall", study.best_value)
    
    # Ré-entraînement sur les meilleurs hyperparamètres trouvés
    best_model = xgb.XGBClassifier(**study.best_params, random_state=42, eval_metric="logloss")
    best_model.fit(X_train, y_train)
    
    # Sauvegarde de ce champion dans l'artefact (et plus tard le registre !)
    mlflow.xgboost.log_model(best_model, "XGBoost_Optuna_Best")
    
    print("\n--- Résultats Optuna ---")
    print(f"✅ Optimisation terminée. Meilleur Recall : {study.best_value:.4f}")
    print("Modèle 'XGBoost_Optuna_Best' enregistré dans MLflow.")

## 4. Bilan du Comparatif

Ouvrez le **Dashboard MLflow** (http://127.0.0.1:5000).

1. Côté **GridSearch**, l'autolog a inondé notre experiment, mais a identifié un bon Recall sur la base de notre grille rigide.
2. Côté **Optuna**, regardez le run `Optuna_Master_Run`. Ouvrez-le. Vous verrez que les sous-runs (les *trials*) sont encapsulés proprement.

En comparant les valeurs, **Optuna** trouve très souvent une meilleure configuration en bien moins d'essais en naviguant astucieusement dans un espace de recherche complexe.

### Étape finale pour le développeur (À vous de jouer !) :
Allez dans l'UI MLflow, cliquez sur le meilleur modèle issu d'Optuna, et **enregistrez-le dans le Model Registry** en remplaçant la version de notre précédent carnet (`Detecteur_Accidents_Route`). Vous venez de déployer votre version la plus intelligente !